*Imports & Basic Paths*

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from glob import glob
import os
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd

DATA_ROOT = Path(r"D:\Dataset")
YAML_PATH = DATA_ROOT / "plantdoc.yaml"
WEIGHTS_PATH = DATA_ROOT / "yolov8n.pt"

print("DATA_ROOT:", DATA_ROOT)
print("YAML_PATH:", YAML_PATH)
print("WEIGHTS_PATH:", WEIGHTS_PATH)

print("Train images dir:", DATA_ROOT / "train" / "images")
print("Test images dir:", DATA_ROOT / "test" / "images")

*Detect Number of Classes*

In [ ]:
labels_root_train = DATA_ROOT / "train" / "labels"
labels_root_test = DATA_ROOT / "test" / "labels"

label_files = glob(str(labels_root_train / "*.txt")) + \
              glob(str(labels_root_test / "*.txt"))

print("Total label files found:", len(label_files))
print("First few label files:")
for lf in label_files[:5]:
    print("  ", lf)

cls_ids = set()

for lf in label_files:
    with open(lf, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            try:
                cls = int(parts[0])
                cls_ids.add(cls)
            except ValueError:
                pass

print("\nUnique class ids:", sorted(cls_ids))

if not cls_ids:
    raise RuntimeError("No Class ID Found!")

max_cls = max(cls_ids)
nc = max_cls + 1
print("Max class id:", max_cls)
print("=> nc should be:", nc)

*Update plantdoc.yaml*

In [ ]:
cfg = {
    "path": "D:/Dataset",
    "train": "train/images",
    "val": "test/images",
    "test": "test/images",
    "nc": int(nc),
    "names": [f"class_{i}" for i in range(int(nc))],
}

print("Saving YAML to:", YAML_PATH)

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

print("✅ plantdoc.yaml written.")
print(cfg)

*Check Paths & Load Pretrained Model*

In [ ]:
with open(YAML_PATH, "r", encoding="utf-8") as f:
    cfg_loaded = yaml.safe_load(f)

print("Loaded from YAML:")
print(cfg_loaded)

root = cfg_loaded["path"]
train_dir = os.path.join(root, cfg_loaded["train"])
val_dir = os.path.join(root, cfg_loaded["val"])

print("\nTrain dir:", train_dir, "exists:", os.path.exists(train_dir))
print("Val dir:", val_dir, "exists:", os.path.exists(val_dir))

assert WEIGHTS_PATH.exists(), f"Weights file not found: {WEIGHTS_PATH}"

model = YOLO(str(WEIGHTS_PATH))
print("\n✅ Pretrained model loaded:", WEIGHTS_PATH)
